# 04 — rqm-core as the Source of Truth

In the previous notebooks we built up intuition using illustrative Python snippets.  
Now it's time to use the **real tool**: `rqm-core`.

`rqm-core` is the canonical math layer of the RQM ecosystem.  
**All quaternion, spinor, Bloch sphere, and SU(2) logic lives there.**  
Notebooks import from it; they never reimplement it.

---

## The architectural rule

```
rqm-core  ──►  rqm-qiskit  ──►  rqm-notebooks
   ▲
   │
   source of truth
```

- ✅ **Do**: `from rqm_core import Quaternion, Spinor, SU2`
- ❌ **Don't**: reimplement quaternion multiplication, Bloch conversion, etc. in a notebook


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.display_utils import show_matrix, show_state_vector, show_info_table
from helpers.plotting import draw_bloch_sphere
import matplotlib.pyplot as plt
import numpy as np
setup_notebook()


## Importing rqm-core

```python
import rqm_core
from rqm_core import Quaternion, Spinor, BlochVector, SU2
```

The cells below show the *intended usage pattern*.  
If `rqm-core` is not yet installed in your environment, the cells fall back to a  
clearly-labelled illustrative implementation so this notebook remains runnable.


In [ ]:
# Try to import rqm-core; fall back to illustrative stubs if not installed
try:
    import rqm_core
    from rqm_core import Quaternion, Spinor, SU2
    USING_RQM_CORE = True
    print(f"rqm-core {rqm_core.__version__} loaded ✓")
except ImportError:
    USING_RQM_CORE = False
    print("rqm-core not installed — running illustrative stubs.")
    print("Install with: pip install rqm-core")


## Quaternion operations via rqm-core

The `Quaternion` class in `rqm-core` provides all standard operations:
- `Quaternion(w, x, y, z)` — construction
- `q1 * q2` — Hamilton product
- `q.conjugate()` — conjugate
- `q.norm()` — norm
- `q.to_rotation_matrix()` — 3×3 rotation matrix
- `q.to_su2()` — 2×2 SU(2) matrix


In [ ]:
if USING_RQM_CORE:
    # --- using rqm-core ---
    q1 = Quaternion.from_axis_angle([0, 0, 1], angle=np.pi / 2)
    q2 = Quaternion.from_axis_angle([1, 0, 0], angle=np.pi / 4)
    q_composed = q2 * q1
    print("Composed rotation (rqm-core):")
    print(f"  {q_composed}")
    print(f"  norm = {q_composed.norm():.6f}")
else:
    # --- illustrative stub ---
    def quat_from_axis_angle(axis, angle):
        n = np.array(axis, dtype=float)
        n /= np.linalg.norm(n)
        w = np.cos(angle / 2)
        xyz = np.sin(angle / 2) * n
        return np.array([w, *xyz])

    def quat_mul(q1, q2):
        w1,x1,y1,z1 = q1; w2,x2,y2,z2 = q2
        return np.array([
            w1*w2 - x1*x2 - y1*y2 - z1*z2,
            w1*x2 + x1*w2 + y1*z2 - z1*y2,
            w1*y2 - x1*z2 + y1*w2 + z1*x2,
            w1*z2 + x1*y2 - y1*x2 + z1*w2,
        ])

    q1 = quat_from_axis_angle([0, 0, 1], np.pi / 2)
    q2 = quat_from_axis_angle([1, 0, 0], np.pi / 4)
    q_composed = quat_mul(q2, q1)
    norm = np.linalg.norm(q_composed)
    print("Composed rotation (illustrative stub — use rqm-core in production):")
    print(f"  w={q_composed[0]:.4f}, x={q_composed[1]:.4f}, "
          f"y={q_composed[2]:.4f}, z={q_composed[3]:.4f}")
    print(f"  norm = {norm:.6f}")


## Spinor and Bloch conversion via rqm-core

```python
spinor = Spinor(alpha, beta)
bloch  = spinor.to_bloch()   # BlochVector with .x .y .z
```

`rqm-core` handles all normalisation, phase conventions, and edge cases internally.


In [ ]:
if USING_RQM_CORE:
    spinor = Spinor(1 / np.sqrt(2), 1 / np.sqrt(2))   # |+>
    bloch = spinor.to_bloch()
    print(f"Spinor |+> → Bloch vector: ({bloch.x:.4f}, {bloch.y:.4f}, {bloch.z:.4f})")
else:
    def spinor_to_bloch(a, b):
        a, b = complex(a), complex(b)
        norm = np.sqrt(abs(a)**2 + abs(b)**2)
        a, b = a/norm, b/norm
        return np.array([
            2*(a.conjugate()*b).real,
            2*(a.conjugate()*b).imag,
            abs(a)**2 - abs(b)**2,
        ])

    bv = spinor_to_bloch(1/np.sqrt(2), 1/np.sqrt(2))
    print(f"Spinor |+> → Bloch vector (stub): ({bv[0]:.4f}, {bv[1]:.4f}, {bv[2]:.4f})")
    print("→ Use rqm-core Spinor.to_bloch() in production.")


## Why this matters

Centralising math in `rqm-core` means:

1. **Correctness** — one implementation, thoroughly tested
2. **Consistency** — notebooks, scripts, and production code all agree
3. **Maintainability** — fix a bug once, benefit everywhere
4. **Trust** — users can rely on `rqm-core` as the ground truth

The notebooks in this series are demonstrations, not reimplementations.


## Summary

- `rqm-core` owns: `Quaternion`, `Spinor`, `BlochVector`, `SU2`
- Notebooks import from `rqm-core`, they don't reimplement math
- The `helpers/` package provides only display utilities
- This separation keeps the stack clean, correct, and maintainable

**Next:** [05_rqm_qiskit_single_qubit_workflows.ipynb](05_rqm_qiskit_single_qubit_workflows.ipynb)
